In [1]:
!pip install zss
!pip install antlr4-python3-runtime==4.11

  Preparing metadata (setup.py) ... done
  Created wheel for zss: filename=zss-1.2.0-py3-none-any.whl size=6725 sha256=256d2d6981aac3ddf62498ecef454b69daf9ed9dd50e1921534d3202de7d3275
  Stored in directory: /root/.cache/pip/wheels/46/e7/2e/44fb39352ad468427a7528cacbefefaa438a898dfd1ad2eaa4
Successfully built zss
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.9.3
    Uninstalling antlr4-python3-runtime-4.9.3:
      Successfully uninstalled antlr4-python3-runtime-4.9.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.


In [2]:
import os

repo_url = "https://github.com/smiling-demon/llm-symbolic-ode-reasoning.git"
repo_dir = "llm-symbolic-ode-reasoning"

if not os.path.exists(repo_dir):
    !git clone {repo_url}

os.chdir(repo_dir)

!pwd

Cloning into 'llm-symbolic-ode-reasoning'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 65 (delta 25), reused 55 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 56.58 KiB | 2.18 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/kaggle/working/llm-symbolic-ode-reasoning


In [3]:
import pandas as pd

from llm import LLM
from methods import cot
from metrics import evaluate_predictions

In [4]:
df = pd.read_excel("/kaggle/input/datasets/dmitrylessy/ode-basic-dataset/data/test.xlsx")

equations = df['equation'].tolist()
solutions = df['solution'].tolist()

In [5]:
llm = LLM("Qwen/Qwen2.5-3B-Instruct")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
%%time
batch_size = 20
predictions = []

for i in range(0, len(equations), batch_size):
    predictions += cot(llm, equations[i: i + batch_size], max_new_tokens=1024)

CPU times: user 8min 29s, sys: 1.22 s, total: 8min 30s
Wall time: 8min 30s


In [7]:
evaluation = evaluate_predictions(predictions, solutions)
for key in evaluation:
    evaluation[key] = [evaluation[key]]
pd.DataFrame.from_dict(evaluation)

,bleu,ast_error_size,ast_tree_distance,exact_match
0,0.520331,1.491716,0.681276,0.47
